# Variational Rho-Posteriors: Numerical Experiments

This notebook reproduces all figures from:

> M. Khribch & P. Alquier (2025). *Robust Bayesian Inference via Variational
> Approximations of Generalised Rho-Posteriors.* Submitted to JASA.

**Instructions:**
1. Install the package: `pip install -e .` from the project root.
2. Run cells sequentially. Each section runs one experiment and generates its figures.
3. CSV results are saved to `results/` for the R plotting scripts.

**Runtime:** Full execution with T=1000 trials takes several hours on CPU.
Set `N_TRIALS = 50` below for a quick test run.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt

from src.evaluation import (
    evaluate_gaussian,
    evaluate_poisson,
    evaluate_uniform,
    evaluate_fourier_regression,
    evaluate_correlated_regression,
)
from src.plotting import (
    set_paper_style,
    plot_posterior_risk,
    plot_rmse,
    plot_density_gaussian,
    plot_density_poisson,
    plot_density_uniform,
    plot_predicted_vs_true,
)

from pathlib import Path

RESULTS_DIR = Path('../results')
FIGURES_DIR = Path('../figures')
RESULTS_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)

set_paper_style()
%matplotlib inline

# -------------------------------------------------------
# Global settings
# -------------------------------------------------------
N_TRIALS = 1000    # Set to 50 for a quick test run
SEED = 0

np.random.seed(SEED)
torch.manual_seed(SEED)

print(f'Trials per configuration: {N_TRIALS}')
print(f'Device: {"cuda" if torch.cuda.is_available() else "cpu"}')

---
## 1. Gaussian Location Model (Figure 1)

Model: $X_i \sim N(\theta, 1)$ with $\theta^\star = 0$.

Contamination: $P^\star_\varepsilon = (1 - \varepsilon) N(0,1) + \varepsilon N(8,1)$.

In [ ]:
gauss_summary, gauss_trials = evaluate_gaussian(
    n_samples=200, d=1,
    epsilon_values=[0.0, 0.05, 0.08, 0.10],
    n_trials=N_TRIALS, tau=0.5, prior_std=2.0,
    n_iter_opt=200, n_mc_opt=128,
)

gauss_summary.to_csv(RESULTS_DIR / 'gaussian_summary.csv', index=False)
gauss_trials.to_csv(RESULTS_DIR / 'gaussian_trials.csv', index=False)
gauss_summary

In [ ]:
fig1a = plot_posterior_risk(gauss_summary, save_path=FIGURES_DIR / 'posterior_risk_gaussian.pdf')
plt.show()

fig1b = plot_rmse(gauss_summary, save_path=FIGURES_DIR / 'rmse_gaussian.pdf')
plt.show()

fig1c = plot_density_gaussian(gauss_trials, epsilon=0.10,
                              save_path=FIGURES_DIR / 'density_plot_gaussian.pdf')
plt.show()

---
## 2. Poisson Intensity Model (Figure 2)

Model: $X_i \sim \mathrm{Pois}(\lambda)$ with $\lambda^\star = 3$.

Contamination: $P^\star_\varepsilon = (1 - \varepsilon) \mathrm{Pois}(3) + \varepsilon \mathrm{Pois}(30)$.

In [ ]:
pois_summary, pois_trials = evaluate_poisson(
    n_samples=200, lam0=3.0, lam_out=30.0,
    epsilon_values=[0.0, 0.05, 0.10, 0.20],
    n_trials=N_TRIALS, tau=0.5, prior_std=2.0,
    a_gamma=1.0, b_gamma=1.0,
    n_iter_opt=400, n_mc_opt=128,
)

pois_summary.to_csv(RESULTS_DIR / 'poisson_summary.csv', index=False)
pois_trials.to_csv(RESULTS_DIR / 'poisson_trials.csv', index=False)
pois_summary

In [ ]:
fig2a = plot_posterior_risk(pois_summary, save_path=FIGURES_DIR / 'posterior_risk_pois.pdf')
plt.show()

fig2b = plot_rmse(pois_summary, save_path=FIGURES_DIR / 'rmse_pois.pdf')
plt.show()

fig2c = plot_density_poisson(pois_trials, epsilon=0.10,
                             save_path=FIGURES_DIR / 'density_plot_pois.pdf')
plt.show()

---
## 3. Uniform Scale Model (Figure 3)

Model: $X_i \sim U[0, \theta]$ with $\theta^\star = 1$.

Contamination: $P^\star_\varepsilon = (1 - \varepsilon) U[0,1] + \varepsilon U[101, 102]$.

In [ ]:
unif_summary, unif_trials = evaluate_uniform(
    n_samples=200, t0=1.0,
    epsilon_values=[0.0, 0.05, 0.08, 0.10],
    n_trials=N_TRIALS, tau=0.5, prior_std=2.0,
    a_prior=0.5, alpha_prior=2.0,
    n_iter_opt=400, n_mc_opt=128,
)

unif_summary.to_csv(RESULTS_DIR / 'uniform_summary.csv', index=False)
unif_trials.to_csv(RESULTS_DIR / 'uniform_trials.csv', index=False)
unif_summary

In [ ]:
fig3a = plot_posterior_risk(unif_summary, save_path=FIGURES_DIR / 'posterior_risk_uniform.pdf')
plt.show()

fig3b = plot_rmse(unif_summary, save_path=FIGURES_DIR / 'rmse_uniform.pdf')
plt.show()

fig3c = plot_density_uniform(unif_trials, epsilon=0.10,
                             save_path=FIGURES_DIR / 'density_plot_uniform.pdf')
plt.show()

---
## 4. Fourier Basis Regression (Figure 4)

True function: $f^\star(w) = \sin(2\pi w) + 0.3\cos(6\pi w)$.

Fourier design with $K=6$ frequency components ($p = 13$).
Noise: $(1-\varepsilon)N(0,1) + \varepsilon \cdot \mathrm{Pareto}(6, 2)$.

In [ ]:
fourier_summary, fourier_trials = evaluate_fourier_regression(
    n_samples=200, K=6,
    epsilon_values=[0.0, 0.05, 0.08, 0.10],
    n_trials=N_TRIALS, tau=0.5, prior_std=2.0,
    n_iter_opt=500, n_mc_opt=128,
)

fourier_summary.to_csv(RESULTS_DIR / 'fourier_summary.csv', index=False)
fourier_trials.to_csv(RESULTS_DIR / 'fourier_trials.csv', index=False)
fourier_summary

In [ ]:
fig4a = plot_posterior_risk(fourier_summary, save_path=FIGURES_DIR / 'fourier_bayes_risk.pdf')
plt.show()

fig4b = plot_rmse(fourier_summary, save_path=FIGURES_DIR / 'fourier_rmse.pdf')
plt.show()

---
## 5. Correlated Design Regression (Figure 5)

$n = 100$, $d = 10$ correlated features with Toeplitz covariance
$\Sigma_{jk} = 0.7^{|j-k|}$, sparse $\beta^\star$ with 5 nonzero entries.

Noise: $(1-\varepsilon)N(0,1) + \varepsilon \cdot \mathrm{Pareto}(10, 1.5)$.

In [ ]:
corr_summary, corr_trials, corr_pred = evaluate_correlated_regression(
    n_samples=100, d=10, rho_corr=0.7, sparsity=5,
    epsilon_values=[0.0, 0.05, 0.08, 0.10],
    n_trials=N_TRIALS, tau=0.5, prior_std=2.0,
    n_iter_opt=500, n_mc_opt=128,
)

corr_summary.to_csv(RESULTS_DIR / 'correlated_summary.csv', index=False)
corr_trials.to_csv(RESULTS_DIR / 'correlated_trials.csv', index=False)
if not corr_pred.empty:
    corr_pred.to_csv(RESULTS_DIR / 'correlated_predictions.csv', index=False)
corr_summary

In [ ]:
fig5a = plot_posterior_risk(corr_summary, save_path=FIGURES_DIR / 'posterior_risk_regression.pdf')
plt.show()

fig5b = plot_rmse(corr_summary, save_path=FIGURES_DIR / 'rmseregression.pdf')
plt.show()

if not corr_pred.empty:
    fig5c = plot_predicted_vs_true(corr_pred, epsilon=0.10,
                                   save_path=FIGURES_DIR / 'predictedvsfitted.pdf')
    plt.show()

---
## Summary

All results have been saved to `results/` as CSV files.
All figures have been saved to `figures/` as PDFs.

For final publication-quality figures, run the R scripts:
```bash
cd ..  # go to project root
Rscript R/plot_gaussian.R
Rscript R/plot_poisson.R
Rscript R/plot_uniform.R
Rscript R/plot_regression.R
```